# 14 — Keyword Extraction

Given a document, which words or phrases carry the signal? Keyword extraction ranks the vocabulary so the top terms are the ones a human — or an ATS — would use to describe the text. Three families are shown: TF-IDF (frequency statistics), RAKE (phrase scoring), and KeyBERT (neural embeddings).

**Why it matters for resumes / ATS:** a resume's keywords are its ATS currency. Extraction gives you a ranked, reviewable list of what the document is "about" — what matching engines use to decide whether a resume fits a job description, and what human screeners scan for in seconds.

**Goal:** Extract the most important words/phrases from a document automatically.

The extracted keywords are the first unsupervised feature set that can stand in for the whole document: instead of matching against 300 raw tokens, an ATS matches against the 5–10 highest-ranked phrases. This chapter compares a statistical method (TF-IDF), a rule-based one (RAKE), and a semantic one (KeyBERT).

## 1. TF-IDF Keyword Extraction

TF-IDF (detailed in Ch. 16) rewards terms that appear often in one document but rarely across the corpus. Ranked TF-IDF scores are a fast, interpretable keyword extractor with zero training.

**What the code does:** `TfidfVectorizer(stop_words="english", max_features=20)` builds the document-term matrix over three resume-like docs, sums each column, and sorts term-score pairs.
- "learning" tops the list at `0.764`, then "machine" and "python" at `0.682` — the technical core of the corpus.
- Generic words are suppressed: "experience" scores only `0.471` despite appearing in a doc, because IDF punishes corpus-wide terms.

**Try it:** the ranking reads like a one-line resume summary — that's the whole point. Change `max_features` to see the tail of the ranking.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
docs = [
    "Python developer with machine learning and NLP experience",
    "Data scientist skilled in Python, SQL, and machine learning",
    "ML engineer building deep learning models with TensorFlow",
]
vec = TfidfVectorizer(stop_words="english", max_features=20)
matrix = vec.fit_transform(docs)
scores = np.array(matrix.sum(axis=0)).flatten()
terms = vec.get_feature_names_out()
ranked = sorted(zip(terms, scores), key=lambda x: -x[1])
print("TF-IDF Keywords:")
for term, score in ranked:
    print(f"  {term:15s} {score:.3f}")

TF-IDF Keywords:
  learning        0.764
  machine         0.682
  python          0.682
  developer       0.471
  experience      0.471
  nlp             0.471
  data            0.426
  scientist       0.426
  skilled         0.426
  sql             0.426
  building        0.397
  deep            0.397
  engineer        0.397
  ml              0.397
  models          0.397
  tensorflow      0.397


## 2. RAKE (Rapid Automatic Keyword Extraction)

RAKE is rule-based: split text into candidate phrases on punctuation, drop stopwords and short words, then score what's left. It needs no training data and no external model — just a stopword list.

**What the code does:** `rake()` splits on punctuation via `re.split(r"[.,!?;:()]", ...)`, filters tokens by stopwords and length, and returns the most frequent surviving phrases via `Counter`.
- On "This candidate has strong Python and machine learning skills..." it returns `[('this candidate has strong python machine learning skills with deep learning expertise', 1)]`.

**Try it:** this naive implementation only splits on punctuation, so a punctuation-free sentence collapses into one giant "keyword". The classic RAKE improvement is to also split on stopwords, which yields `['strong python', 'machine learning skills', 'deep learning expertise']`-style phrases. Add that split and re-run.

In [2]:
# Simple RAKE implementation
import re
from collections import Counter, defaultdict
def rake(text, stopwords=None):
    if stopwords is None:
        stopwords = {"the", "a", "an", "and", "or", "of", "in", "to", "for", "is", "on", "at"}
    phrases = re.split(r"[.,!?;:()]", text.lower())
    candidates = []
    for phrase in phrases:
        words = [w for w in phrase.split() if w not in stopwords and len(w) > 2]
        if words:
            candidates.append(" ".join(words))
    return Counter(candidates).most_common(10)

text = "This candidate has strong Python and machine learning skills with deep learning expertise"
print("RAKE keywords:", rake(text))

RAKE keywords: [('this candidate has strong python machine learning skills with deep learning expertise', 1)]


## 3. KeyBERT (BERT-based Keyword Extraction)

KeyBERT embeds the document and candidate phrases with a transformer model (`all-MiniLM-L6-v2` here) and returns the candidates with the highest cosine similarity to the document — keywords by *meaning*, not by frequency.

**What the code does:** wraps the call in `try/except` so the notebook survives environments where `keybert` (or its model) is missing.
- With KeyBERT installed: `extract_keywords(text, keyphrase_ngram_range=(1, 2), top_n=5)` prints the top 5 scored phrases.
- Without it: prints "KeyBERT not installed. Install with: pip install keybert" and points back to the TF-IDF cell — the graceful-degradation pattern you want in real pipelines too.

**Try it:** compare KeyBERT's top phrases with TF-IDF's on the same text — the two often agree on the head of the list but diverge in the tail, which is why this chapter's summary says to use both.

## Summary: TF-IDF = fast & interpretable. KeyBERT = semantic but slower. Use both.

**There is no single best keyword extractor — pick by constraint: TF-IDF when you need speed and auditability, KeyBERT when you need meaning, both when you can afford it.**

TF-IDF is deterministic, dependency-free, and explainable ("learning ranked first at 0.764"); KeyBERT understands synonyms ("ML" ≈ "machine learning") but loads a transformer and is slower per document. In an ATS, TF-IDF-style extraction is typically the first pass — and the extracted keywords become the features that Ch. 15–16 vectorize next.